# IBM Applied Data Science Capstone — SpaceX Falcon 9 Landing Prediction
**Prepared for:** Priti Gaikwad

This notebook is designed to run in **Google Colab**. It downloads the course dataset, performs data checks, exploratory visualization, SQL analysis, launch-site mapping, and classification modeling. It then generates `Data Science Capstone Project Report.pdf` from the results produced by this run.

**Important:** Enter your own GitHub repository URL in the configuration cell after you create/upload your project repository. Do not submit another student's repository as your own. All reported metrics are calculated when the notebook runs; do not type in guessed scores.


## 1. Setup and configuration

In [ ]:
%pip -q install pandas numpy matplotlib seaborn plotly folium scikit-learn reportlab requests


In [ ]:
import os, warnings, sqlite3, textwrap, json, math, requests
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from IPython.display import display, IFrame
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

OUT = Path('/content/spacex_capstone_output')
OUT.mkdir(exist_ok=True, parents=True)
DATA = OUT / 'data'
DATA.mkdir(exist_ok=True)

# Replace this value with YOUR repository URL before generating the final PDF.
GITHUB_URL = 'ADD_YOUR_OWN_GITHUB_REPOSITORY_URL_HERE'

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)


## 2. Collect course datasets

The IBM course provides prepared CSV files for independent labs. This notebook downloads the same named datasets from a public capstone repository for convenience. The project repository you submit must be your own and should include the notebook and generated outputs.

In [ ]:
RAW = 'https://raw.githubusercontent.com/Ahmed-Nezar/IBM-Data-Science-Capstone-SpaceX/main/'
files = ['dataset_part_1.csv', 'dataset_part_2.csv', 'dataset_part_3.csv',
         'spacex_launch_dash.csv', 'spacex_launch_geo.csv', 'spacex_web_scraped.csv']
download_log = []
for name in files:
    url = RAW + name
    dest = DATA / name
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        dest.write_bytes(r.content)
        download_log.append((name, 'downloaded', len(r.content)))
    except Exception as e:
        download_log.append((name, f'FAILED: {e}', 0))
display(pd.DataFrame(download_log, columns=['file','status','bytes']))
main_path = DATA / 'dataset_part_2.csv'
if not main_path.exists():
    raise FileNotFoundError('Could not download dataset_part_2.csv. Check internet access and rerun this cell.')
df = pd.read_csv(main_path)
print('Dataset shape:', df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12).to_frame('missing_values'))


## 3. Data wrangling and derived fields

In [ ]:
# Standardize date and target fields, remove exact duplicate rows, and inspect data types.
df = df.drop_duplicates().copy()
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
if 'Class' not in df.columns:
    raise ValueError("Expected target column 'Class' is missing from the course dataset.")
df['Class'] = pd.to_numeric(df['Class'], errors='coerce')
df = df.dropna(subset=['Class'])
df['Class'] = df['Class'].astype(int)
if 'Year' not in df.columns and 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year
print('Cleaned shape:', df.shape)
print('Target distribution (0=failure, 1=success):')
display(df['Class'].value_counts().rename_axis('Class').to_frame('Launches'))
print('Missing values after basic cleaning:')
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame('Missing'))


## 4. Exploratory data analysis — visualization results

In [ ]:
# Chart 1: launch outcome distribution
fig, ax = plt.subplots(figsize=(7,4))
df['Class'].map({0:'Unsuccessful landing',1:'Successful landing'}).value_counts().plot(kind='bar', ax=ax)
ax.set_title('Falcon 9 First-Stage Landing Outcomes'); ax.set_ylabel('Number of launches'); ax.set_xlabel('')
plt.xticks(rotation=15, ha='right'); plt.tight_layout()
fig.savefig(OUT/'outcome_distribution.png', dpi=160, bbox_inches='tight'); plt.show()

# Chart 2: launches by site and success
if 'LaunchSite' in df.columns:
    site_rate = df.groupby('LaunchSite')['Class'].agg(['count','mean']).sort_values('mean', ascending=False)
    site_rate['success_pct'] = 100*site_rate['mean']
    display(site_rate)
    fig, ax = plt.subplots(figsize=(8,4))
    (100*df.groupby('LaunchSite')['Class'].mean()).sort_values().plot(kind='barh', ax=ax)
    ax.set_title('First-Stage Landing Success Rate by Launch Site'); ax.set_xlabel('Success rate (%)'); ax.set_ylabel('')
    plt.tight_layout(); fig.savefig(OUT/'site_success_rate.png', dpi=160, bbox_inches='tight'); plt.show()

# Chart 3: payload mass vs flight number
if {'FlightNumber','PayloadMass','Class'}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(8,4.5))
    sns.scatterplot(data=df, x='FlightNumber', y='PayloadMass', hue='Class', alpha=.8, ax=ax)
    ax.set_title('Payload Mass and Flight Number vs Landing Outcome')
    plt.tight_layout(); fig.savefig(OUT/'payload_flight_scatter.png', dpi=160, bbox_inches='tight'); plt.show()

# Chart 4: yearly launch trend
if 'Year' in df.columns:
    yearly = df.groupby('Year').agg(launches=('Class','size'), success_rate=('Class','mean'))
    yearly['success_pct'] = 100*yearly['success_rate']
    display(yearly)
    fig, ax = plt.subplots(figsize=(8,4))
    yearly['launches'].plot(marker='o', ax=ax)
    ax.set_title('Falcon 9 Launches by Year'); ax.set_ylabel('Launch count'); ax.set_xlabel('Year')
    plt.tight_layout(); fig.savefig(OUT/'yearly_launches.png', dpi=160, bbox_inches='tight'); plt.show()

# Chart 5: orbit success rate
if {'Orbit','Class'}.issubset(df.columns):
    orbit = df.groupby('Orbit')['Class'].agg(['count','mean']).sort_values('mean', ascending=False)
    orbit['success_pct'] = orbit['mean']*100
    display(orbit)
    fig, ax = plt.subplots(figsize=(8,4))
    orbit['success_pct'].sort_values().plot(kind='barh', ax=ax)
    ax.set_title('Landing Success Rate by Orbit'); ax.set_xlabel('Success rate (%)'); ax.set_ylabel('')
    plt.tight_layout(); fig.savefig(OUT/'orbit_success_rate.png', dpi=160, bbox_inches='tight'); plt.show()


## 5. EDA using SQL (SQLite)

In [ ]:
# Store the cleaned course dataset in SQLite and run the rubric-relevant queries.
db_path = OUT/'spacex_eda.db'
conn = sqlite3.connect(db_path)
sql_df = df.copy()
if 'Date' in sql_df.columns:
    sql_df['Date'] = sql_df['Date'].astype(str)
sql_df.to_sql('launches', conn, if_exists='replace', index=False)

queries = {
'Launches by launch site': "SELECT LaunchSite, COUNT(*) AS launches FROM launches GROUP BY LaunchSite ORDER BY launches DESC",
'Payload statistics by launch site': "SELECT LaunchSite, COUNT(PayloadMass) AS n, ROUND(AVG(PayloadMass),1) AS avg_payload_kg, ROUND(MAX(PayloadMass),1) AS max_payload_kg FROM launches GROUP BY LaunchSite ORDER BY avg_payload_kg DESC",
'Landing success rate by launch site': "SELECT LaunchSite, COUNT(*) AS launches, ROUND(100.0*AVG(Class),1) AS success_pct FROM launches GROUP BY LaunchSite ORDER BY success_pct DESC",
'Landing success rate by orbit': "SELECT Orbit, COUNT(*) AS launches, ROUND(100.0*AVG(Class),1) AS success_pct FROM launches GROUP BY Orbit ORDER BY success_pct DESC",
'Yearly launch count and success': "SELECT substr(Date,1,4) AS year, COUNT(*) AS launches, ROUND(100.0*AVG(Class),1) AS success_pct FROM launches WHERE Date IS NOT NULL GROUP BY year ORDER BY year"
}
sql_results = {}
for title, query in queries.items():
    print('\n###', title, '\n', query)
    result = pd.read_sql_query(query, conn)
    sql_results[title] = result
    display(result)


## 6. Interactive launch-site map and proximity analysis

In [ ]:
# Launch-site coordinates are supplied in the course dataset. Map launch sites and sample launch records.
map_df = df.dropna(subset=['Latitude','Longitude']).copy() if {'Latitude','Longitude'}.issubset(df.columns) else pd.DataFrame()
if not map_df.empty:
    center = [map_df['Latitude'].mean(), map_df['Longitude'].mean()]
    fmap = folium.Map(location=center, zoom_start=4, tiles='OpenStreetMap')
    if 'LaunchSite' in map_df.columns:
        for site, g in map_df.groupby('LaunchSite'):
            row = g.iloc[0]
            folium.Marker([row['Latitude'], row['Longitude']],
                          popup=f"{site}<br>Records: {len(g)}",
                          tooltip=str(site), icon=folium.Icon(color='blue', icon='rocket', prefix='fa')).add_to(fmap)
    fmap.save(str(OUT/'spacex_launch_sites_map.html'))
    display(fmap)
    print('Saved interactive map:', OUT/'spacex_launch_sites_map.html')
else:
    print('No latitude/longitude columns available in the loaded dataset; map not generated.')

# Haversine distance between launch-site coordinates (approximate proximity comparison).
def haversine_km(lat1, lon1, lat2, lon2):
    R=6371.0
    p1,p2=np.radians(lat1),np.radians(lat2)
    dp=np.radians(lat2-lat1); dl=np.radians(lon2-lon1)
    a=np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))
if not map_df.empty and 'LaunchSite' in map_df.columns:
    sites = map_df.groupby('LaunchSite')[['Latitude','Longitude']].first().reset_index()
    pairs=[]
    for i in range(len(sites)):
        for j in range(i+1,len(sites)):
            pairs.append((sites.loc[i,'LaunchSite'], sites.loc[j,'LaunchSite'],
                          round(haversine_km(sites.loc[i,'Latitude'],sites.loc[i,'Longitude'],
                                             sites.loc[j,'Latitude'],sites.loc[j,'Longitude']),1)))
    proximity = pd.DataFrame(pairs, columns=['site_1','site_2','distance_km']).sort_values('distance_km')
    display(proximity.head(10))


## 7. Plotly-style dashboard visuals

In [ ]:
# Static Plotly charts are saved as HTML and can be used to demonstrate the dashboard visuals.
import plotly.express as px
dash_df = df.copy()
if {'LaunchSite','Class'}.issubset(dash_df.columns):
    success_by_site = dash_df.groupby('LaunchSite', as_index=False)['Class'].mean()
    success_by_site['SuccessRatePct'] = success_by_site['Class']*100
    fig = px.pie(success_by_site, names='LaunchSite', values='SuccessRatePct',
                 title='Distribution of Site-Level Success Rates')
    fig.write_html(str(OUT/'dashboard_success_pie.html'))
    display(fig)
if {'PayloadMass','Class','LaunchSite'}.issubset(dash_df.columns):
    fig2 = px.scatter(dash_df, x='PayloadMass', y='Class', color='LaunchSite',
                      hover_data=['FlightNumber'] if 'FlightNumber' in dash_df.columns else None,
                      title='Payload Mass vs Landing Outcome by Launch Site')
    fig2.write_html(str(OUT/'dashboard_payload_outcome.html'))
    display(fig2)
print('Interactive Plotly charts saved in:', OUT)


## 8. Predictive analysis — compare multiple classifiers

In [ ]:
# Avoid leakage: exclude direct outcome text and the target itself; use features available before landing.
target = 'Class'
exclude = {target, 'Outcome', 'LandingPad', 'Serial', 'Date', 'Year'}
feature_cols = [c for c in df.columns if c not in exclude]
X = df[feature_cols].copy()
y = df[target].astype(int).copy()

# Drop columns with all values missing and columns with only one unique non-null value.
X = X.dropna(axis=1, how='all')
X = X.loc[:, X.nunique(dropna=True) > 1]
numeric_features = X.select_dtypes(include=['number','bool']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', numeric_pipe, numeric_features),
                                ('cat', categorical_pipe, categorical_features)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Support Vector Machine': SVC(kernel='rbf', class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=5),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}
fitted, rows, cms = {}, [], {}
for name, estimator in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0)
    })
    cms[name] = confusion_matrix(y_test, pred, labels=[0,1])
metrics = pd.DataFrame(rows).sort_values(['F1','Accuracy'], ascending=False)
display(metrics.style.format({'Accuracy':'{:.3f}','Precision':'{:.3f}','Recall':'{:.3f}','F1':'{:.3f}'}))
metrics.to_csv(OUT/'model_metrics.csv', index=False)
best_name = metrics.iloc[0]['Model']
best_pipe = fitted[best_name]
best_pred = best_pipe.predict(X_test)
print('Best model selected by test F1:', best_name)
print(classification_report(y_test, best_pred, zero_division=0))
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cms[best_name], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted failure','Predicted success'],
            yticklabels=['Actual failure','Actual success'], ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}'); ax.set_xlabel('Prediction'); ax.set_ylabel('Actual')
plt.tight_layout(); fig.savefig(OUT/'best_model_confusion_matrix.png', dpi=160, bbox_inches='tight'); plt.show()
fig, ax = plt.subplots(figsize=(8,4))
metrics.set_index('Model')[['Accuracy','Precision','Recall','F1']].plot(kind='bar', ax=ax)
ax.set_ylim(0,1); ax.set_title('Classifier Evaluation Comparison'); ax.set_ylabel('Score'); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); fig.savefig(OUT/'model_comparison.png', dpi=160, bbox_inches='tight'); plt.show()


## 9. Generate the PDF report from the actual run

In [ ]:
from reportlab.lib.pagesizes import landscape, letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import inch

pdf_path = OUT/'Data Science Capstone Project Report.pdf'
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name='CoverTitle', parent=styles['Title'], fontSize=26, leading=31, alignment=TA_CENTER, textColor=colors.HexColor('#17324D'), spaceAfter=18))
styles.add(ParagraphStyle(name='Section', parent=styles['Heading1'], fontSize=18, leading=22, textColor=colors.HexColor('#17324D'), spaceBefore=5, spaceAfter=10))
styles.add(ParagraphStyle(name='SmallBody', parent=styles['BodyText'], fontSize=9, leading=13, spaceAfter=5))
story=[]
def P(txt, style='SmallBody'):
    story.append(Paragraph(txt, styles[style] if style in styles else styles['SmallBody']))
def add_img(filename, width=8.8*inch):
    p=OUT/filename
    if p.exists():
        im=RLImage(str(p)); im.drawWidth=width; im.drawHeight=width*0.52; story.append(im); story.append(Spacer(1,8))
def add_table(frame, max_rows=12):
    if frame is None or frame.empty: return
    data=[[str(c) for c in frame.columns]]
    for _,r in frame.head(max_rows).iterrows():
        data.append([str(x)[:70] for x in r.tolist()])
    t=Table(data, repeatRows=1, hAlign='LEFT')
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#17324D')),
                           ('TEXTCOLOR',(0,0),(-1,0),colors.white),('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
                           ('FONTSIZE',(0,0),(-1,-1),7),('GRID',(0,0),(-1,-1),.3,colors.grey),
                           ('VALIGN',(0,0),(-1,-1),'TOP'),('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,colors.HexColor('#F1F5F9')])]))
    story.append(t); story.append(Spacer(1,8))

P('DATA SCIENCE CAPSTONE PROJECT REPORT','CoverTitle')
P('SpaceX Falcon 9 First-Stage Landing Prediction','Heading2')
P('Prepared for: Priti Gaikwad')
P('Business question: Can historical launch, payload, orbit, and launch-site characteristics help estimate whether a Falcon 9 first stage will land successfully?')
P('Repository URL: '+GITHUB_URL.replace('&','&amp;'))
P('Dataset source: IBM course-style SpaceX launch datasets hosted in a public repository; the project code downloads the CSV files at runtime.')
story.append(PageBreak())

P('Executive Summary','Section')
success_pct = 100*df['Class'].mean()
P(f'This analysis uses {len(df)} cleaned launch records. The observed first-stage landing success share in the loaded dataset is {success_pct:.1f}%. The workflow combines data wrangling, visual EDA, SQL aggregation, launch-site mapping, interactive Plotly charts, and four classification algorithms. The model comparison and confusion matrix below are generated from the current notebook run, using a held-out test set.')
P('The model scores are predictive estimates for this dataset and split, not guarantees of future mission outcomes. Operational use would require temporal validation, leakage review, and ongoing monitoring.')
story.append(PageBreak())

P('Introduction and Problem Definition','Section')
P('Reusable first stages can reduce launch costs, making landing outcomes relevant to launch planning and commercial bidding. The target variable Class is 1 for a successful landing and 0 for an unsuccessful landing. The objective is to explore patterns in launch history and compare models that classify the landing outcome.')
P('Data Collection — SpaceX API','Section')
P('The course workflow uses the SpaceX API to collect launch records. Typical API workflow: request launch endpoints → parse JSON responses → extract launch date, rocket, payload, launchpad, cores, and landing outcomes → normalize nested fields → save tabular records. API reference: https://api.spacexdata.com/v4/launches')
P('Data Collection — Web Scraping','Section')
P('The companion scraping workflow requests the Wikipedia Falcon 9 and Falcon Heavy launch-history table, parses HTML tables with BeautifulSoup/pandas, normalizes column names, and exports a CSV for cross-checking. Scraped records are useful for historical context; the predictive model below uses the structured course dataset.')
P('Data Wrangling Methodology','Section')
P('The notebook removes duplicate rows, parses dates, checks missing values, coerces the binary target, and reports remaining missingness. Numeric missing values are imputed with the median; categorical values are imputed with the most frequent category. Categorical predictors are one-hot encoded and numeric predictors standardized inside a pipeline.')
story.append(PageBreak())

P('EDA with Data Visualization','Section')
P('Charts produced: outcome distribution, launch-site success rate, payload mass versus flight number, annual launch trend, and orbit-level success rate. The charts show how the observed dataset varies across mission history and categories; they should be interpreted descriptively rather than as causal proof.')
add_img('outcome_distribution.png', 7.5*inch)
add_img('site_success_rate.png', 7.5*inch)
story.append(PageBreak())

P('EDA with Data Visualization — Additional Results','Section')
add_img('payload_flight_scatter.png', 7.5*inch)
add_img('yearly_launches.png', 7.5*inch)
add_img('orbit_success_rate.png', 7.5*inch)
story.append(PageBreak())

P('EDA with SQL Queries','Section')
P('The cleaned data is stored in SQLite. Queries summarize launch counts by site, payload statistics by site, success rate by site, success rate by orbit, and yearly launch counts/success rates.')
for title, result in sql_results.items():
    P(title, 'Heading3')
    add_table(result, 10)
story.append(PageBreak())

P('Interactive Visual Analytics — Folium','Section')
P('The notebook builds an interactive Folium map with markers at the latitude/longitude coordinates included in the course dataset. The proximity table uses the Haversine formula to estimate pairwise straight-line distances between site coordinates. These distances are geographic approximations, not road distances.')
if 'proximity' in globals() and not proximity.empty:
    add_table(proximity, 10)
P('Map HTML output: spacex_launch_sites_map.html (generated in the output folder).')
P('Interactive Visual Analytics — Plotly','Section')
P('The notebook exports interactive Plotly HTML charts: a pie chart comparing site-level success rates and a payload-versus-outcome scatter plot. These provide hover details and can be opened in a browser. A full deployed Dash server is not automatically hosted by this notebook.')
P('Outputs: dashboard_success_pie.html and dashboard_payload_outcome.html.')
story.append(PageBreak())

P('Predictive Analysis — Classification','Section')
P(f'The notebook compares Logistic Regression, Support Vector Machine, Decision Tree, and K-Nearest Neighbors using a stratified 75/25 train/test split. Numeric and categorical preprocessing is fitted only on training data within each pipeline.')
add_table(metrics.round(3), 8)
P(f'Best model by test F1 in this run: <b>{best_name}</b>. Its confusion matrix is shown below. Scores depend on the dataset version, preprocessing, and split; they must be read from this executed run.')
add_img('best_model_confusion_matrix.png', 6.4*inch)
add_img('model_comparison.png', 7.8*inch)
story.append(PageBreak())

P('Conclusion and Recommendations','Section')
P(f'The dataset contains {len(df)} usable records after basic cleaning, with an observed landing success rate of {success_pct:.1f}%. Site, orbit, payload, and launch-history features provide useful exploratory signals. The selected classifier ({best_name}) is the strongest of the four models by F1 on this held-out split.')
P('Recommendations: (1) validate with a time-based split to reduce optimistic estimates; (2) inspect class balance and false negatives/false positives before using predictions operationally; (3) avoid post-outcome fields as predictors; (4) refresh the dataset as new launches occur; (5) treat the model as decision support, not a substitute for engineering assessment.')
P('Reproducibility and GitHub','Section')
P('Before submission, replace the repository placeholder in the configuration cell with your own public GitHub repository URL. Upload the notebook, requirements or setup instructions, generated data/outputs as permitted, and this PDF. Do not claim a repository that belongs to another student.')
P('Source notes','Heading2')
P('Course-style dataset source: https://github.com/Ahmed-Nezar/IBM-Data-Science-Capstone-SpaceX (used as a downloadable dataset source only). SpaceX API reference: https://api.spacexdata.com/v4/launches. IBM course instructions and grading rubric were provided by the learner.')
doc = SimpleDocTemplate(str(pdf_path), pagesize=landscape(letter), rightMargin=0.45*inch, leftMargin=0.45*inch, topMargin=0.45*inch, bottomMargin=0.45*inch)
doc.build(story)
print('PDF generated:', pdf_path)
print('Output folder:', OUT)
print('IMPORTANT: this PDF is only valid after all cells have run successfully and the GitHub URL placeholder is replaced.')


## 10. Submission checklist
1. Run every cell from top to bottom without errors.
2. Inspect generated charts, SQL tables, model metrics, and confusion matrix.
3. Create your own public GitHub repository and upload the notebook and project files.
4. Replace `ADD_YOUR_OWN_GITHUB_REPOSITORY_URL_HERE` with your repository URL and rerun the PDF-generation cell.
5. Download `Data Science Capstone Project Report.pdf` from `/content/spacex_capstone_output/` and upload that PDF to Coursera.
